# Iris Classification Pipeline

End-to-end ML pipeline that trains two tree-based classifiers — **Random Forest** and **XGBoost** — on the Iris dataset, evaluates them comprehensively, and exports a single JSON artifact consumed by the React dashboard.

## Why these two models?

| | Random Forest | XGBoost |
|---|---|---|
| **Strategy** | Bagging (parallel trees, vote) | Boosting (sequential trees, correct errors) |
| **Bias-Variance** | Reduces variance | Reduces both bias and variance |
| **Speed** | Fast to train | Slower but often higher accuracy |
| **Interpretability** | Feature importances via impurity | Feature importances via gain |

Iris is intentionally easy — the goal is to compare model behavior and evaluation structure, not to chase the leaderboard.

In [1]:
import json
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.25
ARTIFACTS_DIR = Path("..") / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"scikit-learn {sklearn.__version__}  |  xgboost {xgboost.__version__}")

scikit-learn 1.7.2  |  xgboost 3.2.0


## 1. Load the Iris dataset

In [2]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
class_names = list(iris.target_names)   # ['setosa', 'versicolor', 'virginica']
feature_names = list(iris.feature_names)

print(f"Samples: {len(X)}  |  Features: {len(feature_names)}  |  Classes: {class_names}")
X.describe().round(3)

Samples: 150  |  Features: 4  |  Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,150.000,150.000,150.000,150.000
mean,5.843,3.057,3.758,1.199
std,0.828,0.436,1.765,0.762
min,4.300,2.000,1.000,0.100
25%,5.100,2.800,1.600,0.300
50%,5.800,3.000,4.350,1.300
75%,6.400,3.300,5.100,1.800
max,7.900,4.400,6.900,2.500


## 2. Train / test split

Stratified 75/25 split with a fixed seed — both models train and evaluate on the exact same partition.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train)} samples  |  Test: {len(X_test)} samples")
print("Train class distribution:", dict(y_train.value_counts().sort_index()))
print("Test  class distribution:", dict(y_test.value_counts().sort_index()))

Train: 112 samples  |  Test: 38 samples


Train class distribution: {0: np.int64(38), 1: np.int64(37), 2: np.int64(37)}
Test  class distribution: {0: np.int64(12), 1: np.int64(13), 2: np.int64(13)}


## 3. Train Random Forest

In [4]:
rf_params = dict(n_estimators=200, max_depth=None, min_samples_leaf=1, random_state=RANDOM_STATE)
rf = RandomForestClassifier(**rf_params)
rf.fit(X_train, y_train)
print("Random Forest trained.")

Random Forest trained.


## 4. Train XGBoost

In [5]:
xgb_params = dict(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    use_label_encoder=False,
    random_state=RANDOM_STATE,
)
xgb = XGBClassifier(**xgb_params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("XGBoost trained.")

XGBoost trained.


## 5. Evaluation helper

A single function that computes the full evaluation payload for any classifier with `predict_proba`.

In [6]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, class_names, feature_names, model_id, hyperparameters):
    """Returns a structured evaluation dict matching the JSON schema."""
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)  # shape (n_test, n_classes)

    # --- core metrics ---
    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    ll = log_loss(y_te, y_prob)

    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_te, y_pred, average="macro")
    prec_wtd,   rec_wtd,   f1_wtd,   _ = precision_recall_fscore_support(y_te, y_pred, average="weighted")

    # --- confusion matrix ---
    cm = confusion_matrix(y_te, y_pred).tolist()

    # --- per-class report ---
    report_dict = classification_report(y_te, y_pred, target_names=class_names, output_dict=True)
    per_class = {
        cn: {
            "precision": round(report_dict[cn]["precision"], 4),
            "recall":    round(report_dict[cn]["recall"],    4),
            "f1_score":  round(report_dict[cn]["f1-score"],  4),
            "support":   int(report_dict[cn]["support"]),
        }
        for cn in class_names
    }

    # --- feature importance ---
    importances = model.feature_importances_.tolist()

    # --- per-sample predictions ---
    test_indices = X_te.index.tolist()
    predictions = [
        {
            "index":         int(idx),
            "y_true":        int(yt),
            "y_true_name":   class_names[int(yt)],
            "y_pred":        int(yp),
            "y_pred_name":   class_names[int(yp)],
            "correct":       bool(yt == yp),
            "probabilities": {cn: round(float(p), 4) for cn, p in zip(class_names, probs)},
            "confidence":    round(float(np.max(probs)), 4),
        }
        for idx, yt, yp, probs in zip(test_indices, y_te.values, y_pred, y_prob)
    ]

    return {
        "id":   model_id,
        "name": "Random Forest" if model_id == "random_forest" else "XGBoost",
        "hyperparameters": hyperparameters,
        "metrics": {
            "accuracy":             round(acc,        4),
            "balanced_accuracy":    round(bal_acc,    4),
            "log_loss":             round(ll,         4),
            "precision_macro":      round(prec_macro, 4),
            "recall_macro":         round(rec_macro,  4),
            "f1_macro":             round(f1_macro,   4),
            "precision_weighted":   round(prec_wtd,   4),
            "recall_weighted":      round(rec_wtd,    4),
            "f1_weighted":          round(f1_wtd,     4),
        },
        "confusion_matrix": {
            "matrix":       cm,
            "labels_order": class_names,
        },
        "classification_report": per_class,
        "feature_importance": {
            "features":   feature_names,
            "importance": [round(v, 6) for v in importances],
        },
        "predictions": predictions,
    }

print("Evaluation helper defined.")

Evaluation helper defined.


## 6. Run evaluation

In [7]:
rf_result = evaluate_model(
    model=rf,
    X_tr=X_train, X_te=X_test, y_tr=y_train, y_te=y_test,
    class_names=class_names,
    feature_names=feature_names,
    model_id="random_forest",
    hyperparameters=rf_params,
)

xgb_result = evaluate_model(
    model=xgb,
    X_tr=X_train, X_te=X_test, y_tr=y_train, y_te=y_test,
    class_names=class_names,
    feature_names=feature_names,
    model_id="xgboost",
    hyperparameters={k: v for k, v in xgb_params.items() if k != "use_label_encoder"},
)

print("RF  accuracy:", rf_result["metrics"]["accuracy"])
print("XGB accuracy:", xgb_result["metrics"]["accuracy"])

RF  accuracy: 0.8947
XGB accuracy: 0.9211


## 7. Build comparison table and dataset summary

In [8]:
COMPARISON_METRICS = [
    "accuracy", "balanced_accuracy", "log_loss",
    "f1_macro", "precision_macro", "recall_macro",
    "f1_weighted",
]

comparison = [
    {
        "metric": m,
        "random_forest": rf_result["metrics"][m],
        "xgboost":       xgb_result["metrics"][m],
    }
    for m in COMPARISON_METRICS
]

# Per-class feature summary for scatter/distribution charts
full_df = X.copy()
full_df["class"] = y.map(dict(enumerate(class_names)))

class_summary = {
    cn: {
        feat: {
            "mean": round(float(grp[feat].mean()), 4),
            "std":  round(float(grp[feat].std()),  4),
            "min":  round(float(grp[feat].min()),  4),
            "max":  round(float(grp[feat].max()),  4),
        }
        for feat in feature_names
    }
    for cn, grp in full_df.groupby("class")
}

# Stratified sample (up to 50 rows) for scatter plots
sample_df = (
    full_df.groupby("class", group_keys=False)
    .apply(lambda g: g.sample(min(17, len(g)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)
sample_rows = [
    {**{feat: round(float(row[feat]), 4) for feat in feature_names}, "class": row["class"]}
    for _, row in sample_df.iterrows()
]

print(f"Comparison table: {len(comparison)} metrics  |  Sample rows: {len(sample_rows)}")

Comparison table: 7 metrics  |  Sample rows: 51

## 8. Assemble and export JSON

Writes `artifacts/model_report.json`, the single source of truth for the React dashboard.

In [9]:
report = {
    "schema_version": 1,
    "meta": {
        "dataset":          "iris",
        "sklearn_version":  sklearn.__version__,
        "xgboost_version":  xgboost.__version__,
        "random_state":     RANDOM_STATE,
        "test_size":        TEST_SIZE,
        "n_samples_total":  len(X),
        "n_samples_train":  len(X_train),
        "n_samples_test":   len(X_test),
        "feature_names":    feature_names,
        "class_names":      class_names,
        "generated_at":     datetime.datetime.utcnow().isoformat() + "Z",
    },
    "data": {
        "class_summary": class_summary,
        "sample":        sample_rows,
    },
    "models": [rf_result, xgb_result],
    "comparison": comparison,
}

out_path = ARTIFACTS_DIR / "model_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

size_kb = out_path.stat().st_size / 1024
print(f"Written to {out_path}  ({size_kb:.1f} KB)")

Written to ..\artifacts\model_report.json  (43.5 KB)

## 9. Quick sanity check

Print a human-readable summary to confirm the export is correct before the dashboard loads it.

In [10]:
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
header = f"{'Metric':<25} {'Random Forest':>15} {'XGBoost':>15}"
print(header)
print("-" * 60)
for row in comparison:
    print(f"{row['metric']:<25} {row['random_forest']:>15.4f} {row['xgboost']:>15.4f}")

print()
for model_result in [rf_result, xgb_result]:
    print(f"\n{model_result['name']} confusion matrix (rows=true, cols=pred):")
    cm = model_result["confusion_matrix"]
    print(f"  Classes: {cm['labels_order']}")
    for i, row in enumerate(cm["matrix"]):
        print(f"  {cm['labels_order'][i]:<12}: {row}")

    misclassified = [p for p in model_result["predictions"] if not p["correct"]]
    print(f"  Misclassified: {len(misclassified)}/{len(model_result['predictions'])} test samples")

MODEL COMPARISON SUMMARY
Metric                      Random Forest         XGBoost
------------------------------------------------------------
accuracy                           0.8947          0.9211
balanced_accuracy                  0.8974          0.9231
log_loss                           0.1905          0.2571
f1_macro                           0.8968          0.9230
precision_macro                    0.9030          0.9246
recall_macro                       0.8974          0.9231
f1_weighted                        0.8941          0.9209


Random Forest confusion matrix (rows=true, cols=pred):
  Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
  setosa      : [12, 0, 0]
  versicolor  : [0, 12, 1]
  virginica   : [0, 3, 10]
  Misclassified: 4/38 test samples

XGBoost confusion matrix (rows=true, cols=pred):
  Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
  setosa      : [12, 0, 0]
  versicolor  : [0, 12, 1]
  virginica   : [0, 2